In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import joblib

In [13]:
df = pd.read_csv(r"..\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No


In [14]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [15]:
df.drop('customerID',axis=1,inplace=True)

In [16]:
df['TotalCharges']=df['TotalCharges'].replace(' ',"0")

In [17]:
df.TotalCharges = df.TotalCharges.astype(float)

In [18]:
df['Churn'] = df["Churn"].replace({'Yes':1,'No':0})
df.Churn = df.Churn.astype(int)

In [19]:
cat_col = df.select_dtypes(include='string').columns.to_list()
cat_col

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

### No need to labelEncode the Target column as it is binary | we will just replace

In [20]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [21]:
df.Churn.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [22]:
encoders={}
labelEn = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)
arr = labelEn.fit_transform(df[cat_col])
fr = pd.DataFrame(arr,columns=labelEn.get_feature_names_out(cat_col))
df.drop(cat_col,axis=1,inplace=True)
df = pd.concat([df,fr],axis=1)
with open("../models/encoder.pkl","wb") as f:
    joblib.dump(encoders,f)


In [23]:
for col in df.columns:
    print(col)
    print(df[col].unique())
    print('-'*50)

SeniorCitizen
[0 1]
--------------------------------------------------
tenure
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
--------------------------------------------------
MonthlyCharges
[29.85 56.95 53.85 ... 63.1  44.2  78.7 ]
--------------------------------------------------
TotalCharges
[  29.85 1889.5   108.15 ...  346.45  306.6  6844.5 ]
--------------------------------------------------
Churn
[0 1]
--------------------------------------------------
gender_Female
[1. 0.]
--------------------------------------------------
gender_Male
[0. 1.]
--------------------------------------------------
Partner_No
[0. 1.]
--------------------------------------------------
Partner_Yes
[1. 0.]
--------------------------------------------------
Dependents_No
[1. 0.]
--------------------------------------------------


In [24]:
df.to_csv("../data/processed/customer.csv",index=False)